# L12c: Deep Q-Learning
In this lecture, we replace the Q-table with a function that computes the action values from a few numbers that describe the state. This idea is the basis of deep Q-learning and of its best-known method, the deep Q-network (DQN). We start with the simplest such function, a linear one.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Explain the limits of a Q-table:__ A table needs one entry for every state and action, and the number of states grows exponentially with the number of state variables. We'll also see that a table learns nothing about a state from the states around it.
> * __Write the DQN loss and its gradient step:__ Deep Q-learning fits a Q-network to targets built from its own estimates, by gradient descent on a squared error. We'll derive the gradient step for a linear Q-network and show that tabular Q-learning is a special case of it.
> * __Describe the replay buffer and the target network:__ The replay buffer trains the network on random samples of past experience, and the target network holds the targets fixed between copies. We'll see which problem of Q-learning with a network each one addresses.

Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Learn a dosing policy without a grid using a linear DQN](CHEME-5800-L12c-Example-LinearDQN-AdaptiveDosing-Fall-2026.ipynb). In this example, an agent learns to dose the patient of the dosing lab without rounding the tumor burden and the toxicity to grid levels. The agent's linear Q-network has 148 weights, where the dosing lab's Q-table has 6,724 entries. We compare its doses with value iteration on a much finer grid.

___

## The Curse of Dimensionality
__The Q-table.__ Entry $(i,j)$ holds the value (utility) of taking action $j$ in state $i$. The policy reads each row and picks the column with the largest entry. For state $s$, that is action $k$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="../L12a/figs/Fig-Q-Schematic.svg" width="580" style="max-width:100%; height:auto;" alt="A grid of dots with states as rows and actions as columns. The dot in row state i and column action j is highlighted as utility (i,j). In the row for state s, the dot in column action k is highlighted as the maximum utility for state s."/>
</p>

The Q-table has three problems:

* __The table grows exponentially.__ A state with $m$ variables, each rounded to $L$ levels, has $L^{m}$ states. The dosing lab's patient has two variables on 41 levels, or $41^{2}=1681$ states, but ten variables would give about $1.3\times 10^{16}$ states. Richard Bellman called this growth the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality).
* __The table does not generalize.__ Each entry learns only from visits to its own state-action pair. Two states with almost the same tumor burden share nothing.
* __Rounding changes the problem.__ A coarse grid treats different states as the same state. In the example, the dosing lab's policy gives no dose at a tumor burden of 0.109, because the grid rounds 0.109 to 0.100, a level that counts as a cure.

Instead, we replace the table with a function that computes the action value $Q(s,a)$, the expected discounted return of action $a$ in state $s$, from a few numbers that describe the state.

___

## Q-Networks
A __Q-network__ is a function $Q(s,a;\theta)$ with adjustable parameters $\theta$. It takes a state $s$ as its input and returns one Q-value $Q(s,a;\theta)$ for each of the $n$ actions in the action set $\mathcal{A}=\{a_{1},\dots,a_{n}\}$. The greedy policy picks the action with the largest Q-value, $\pi(s)=\arg\max_{a\in\mathcal{A}}Q(s,a;\theta)$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Q-Networks/Fig-Q-Networks.svg" width="720" style="max-width:100%; height:auto;" alt="Two diagrams, each mapping a state s to one Q-value per action, Q of s and a 1 through Q of s and a n. The top diagram, labeled linear Q-network, sends the state through gray arrows to a column of feature circles phi 1, phi 2, down to phi p, labeled features phi of s, fixed. Blue lines, labeled weights W, learned, connect every feature to every Q-value. The bottom diagram, labeled deep Q-network, sends the state into a shaded box of three hidden layers of circles joined by blue lines, labeled hidden layers, parameters theta, learned. One arrow leaves the box and fans out to the Q-values."/>
</p>

__Two Q-networks.__ In the linear network, the $p$ features $\phi_{1}(s),\dots,\phi_{p}(s)$ are fixed functions of the state, and only the weights (blue) are learned. In the deep network, the hidden layers between the state and the Q-values are learned as well.

Learning now means adjusting the (unknown) parameter vector $\theta$ instead of the entries of a table. The number of parameters is set by the network, not by the number of states.

### A linear Q-network
A linear Q-network describes the state $s$ by a __feature vector__ $\phi(s)\in\mathbb{R}^{p}$, a list of $p$ numbers computed from the state. The Q-value of action $a$, a weighted sum of the features with weight vector $\mathbf{w}_{a}\in\mathbb{R}^{p}$, is given by:

$$
\begin{equation*}
Q(s,a;\mathbf{W}) = \mathbf{w}_{a}^{\top}\phi(s) = \sum_{j=1}^{p}w_{a,j}\,\phi_{j}(s),
\end{equation*}
$$

where $w_{a,j}$ is the weight of action $a$ on feature $\phi_{j}(s)$. The weight matrix $\mathbf{W}$ has one row $\mathbf{w}_{a}^{\top}$ per action, which gives $n\cdot p$ weights in all.

__The features.__ In the example, the state is the pair $s=(B,Z)$ of tumor burden and toxicity. Its features are __radial basis functions__, with centers $\mathbf{c}_{j}$ spread over the square of states. Feature $j$ is given by:

$$
\begin{equation*}
\phi_{j}(s)=\exp\left(-\frac{\lVert s-\mathbf{c}_{j}\rVert_{2}^{2}}{2\sigma^{2}}\right),
\end{equation*}
$$

where $\lVert s-\mathbf{c}_{j}\rVert_{2}$ is the Euclidean distance from the state to center $j$, and $\sigma$ sets the width of every basis function. Basis function $j$ equals one at its center and falls toward zero away from it. Because nearby states have similar features and similar Q-values, what the agent learns in one state carries over to its neighbors.

__A Q-table is a linear network.__ Number the states and let $\phi(s)$ be the __one-hot__ vector, with a one in entry $s$ and zeros everywhere else. Then $Q(s,a;\mathbf{W})=w_{a,s}$. The weight matrix $\mathbf{W}$ is the transpose of the Q-table.

### A deep Q-network
A __neural network__ is a function built from layers. A __deep Q-network__ replaces the fixed features with the __hidden layers__ of a neural network, the layers between its input and its output.

Each hidden layer is a column of __units__, the circles in the figure. Each unit takes a weighted sum $x$ of its inputs and passes it through a fixed nonlinear function, such as $\max(0,x)$.

The output layer takes weighted sums only, which lets a Q-value be negative. The hidden layers learn their own features from the data.

[Mnih et al. (2015)](https://www.nature.com/articles/nature14236) trained a deep Q-network to play Atari video games from the pixels on the screen. The rest of this lecture holds for any Q-network with parameters $\theta$, the weights $\mathbf{W}$ in the linear case.

___

## The DQN Loss
Q-learning moves the entry $Q(s,a)$ toward the target $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$, where $r$ is the reward, $s^{\prime}$ is the next state, and $\gamma$ is the discount factor. Deep Q-learning instead adjusts $\theta$ to move the network's prediction $Q(s,a;\theta)$ toward a target of the same form.

The training data are __transitions__ $(s_{i},a_{i},r_{i},s_{i}^{\prime},d_{i})$: a state, the action taken, the reward, the next state, and a __done flag__ $d_{i}$. The flag is one if $s_{i}^{\prime}$ is __absorbing__, a state that ends the episode, such as a cure or a toxic death, and zero otherwise. An episode that stops at a step limit keeps $d_{i}=0$ on its last transition, because its next state still has a future.

> __DQN Bellman Loss:__
>
> The target $y_{i}$ of transition $i$ is the value that the prediction $Q(s_{i},a_{i};\theta)$ should match. It is the reward plus the discounted value of the next state:
>
> $$y_{i} = r_{i} + \gamma\,(1-d_{i})\max_{a^{\prime}\in\mathcal{A}}Q(s_{i}^{\prime},a^{\prime};\theta^{-}),$$
>
> where $\theta^{-}$ are the parameters of the __target network__, a delayed copy of the Q-network that we explain in the next section. The factor $(1-d_{i})$ drops the future value when $s_{i}^{\prime}$ is absorbing, which leaves $y_{i}=r_{i}$.
>
> The difference $y_{i}-Q(s_{i},a_{i};\theta)$ is the __prediction error__ of transition $i$. The network trains on a __mini-batch__, a small random sample of $N_{\text{batch}}$ past transitions. The loss is half the mean of the squared prediction errors:
>
> $$\mathcal{L}(\theta) = \frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\left(y_{i}-Q(s_{i},a_{i};\theta)\right)^{2}.$$
>
> The prediction uses the action $a_{i}$ that was actually taken. The target uses the best next action under the target network.

This fit is a __regression__, with the targets $y_{i}$ as the labels. The targets are called __bootstrap targets__, because they are built from the network's own estimates of the next state's value.

Training uses __gradient descent__. One step with learning rate $\alpha$, which sets the size of the step, is $\theta\gets\theta-\alpha\nabla_{\theta}\mathcal{L}(\theta)$, where the gradient $\nabla_{\theta}\mathcal{L}$ is the vector of derivatives of the loss with respect to the parameters. With a small enough $\alpha$, each step pulls the predictions $Q(s_{i},a_{i};\theta)$ of the mini-batch, as a group, closer to their targets $y_{i}$.

### The gradient step of a linear network
For the linear network, the prediction is $Q(s_{i},a_{i};\mathbf{W})=\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})$. Its derivative with respect to $\mathbf{w}_{a_{i}}$ is the feature vector $\phi(s_{i})$, and the other rows of $\mathbf{W}$ do not appear in it. The target $y_{i}$ does not depend on $\mathbf{W}$, because the target network computes it.

One gradient step updates the row of each action in the mini-batch. The update of row $a$ is given by:

$$
\begin{equation*}
\mathbf{w}_{a}\gets\mathbf{w}_{a}+\frac{\alpha}{N_{\text{batch}}}\sum_{i\,:\,a_{i}=a}\left(y_{i}-\mathbf{w}_{a}^{\top}\phi(s_{i})\right)\phi(s_{i}),
\end{equation*}
$$

where the sum runs over the transitions of the mini-batch that took action $a$. The factor $\tfrac{1}{2}$ in the loss cancels the 2 from the derivative of the square. Every prediction error uses the weights from before the step.

Each transition adds its prediction error times its features. The logistic regression step has the same form.

__A check against the table.__ Take one-hot features, a mini-batch that holds only the transition just taken, and a target network that is always equal to the Q-network. The update then reads $Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)$. This is the tabular Q-learning update.

___

## DQN Algorithm and Stabilizers
Tabular Q-learning converges when every state-action pair that is not absorbing is visited infinitely often. For each pair, the learning rates of its updates must add up to infinity, while their squares add up to a finite number.

Deep Q-learning has no such guarantee, and its values can even grow without bound, a failure called __divergence__ ([Sutton and Barto, 2018, Chapter 11](http://incompleteideas.net/book/the-book-2nd.html)). Thus, deep Q-learning adds two stabilizers that make training much steadier in practice:

* __The replay buffer:__ Consecutive transitions of one episode are alike, and many gradient steps on them fit one region of states and spoil the values elsewhere. The __replay buffer__ $\mathcal{D}$ stores the most recent $M$ transitions, and each training step uses a random mini-batch of $N_{\text{batch}}$ of them. Training waits until the buffer holds $N_{\text{warm}}$ transitions, a period called the __warm-up__.
* __The target network:__ If the Q-network computed both the prediction and the target $y_{i}$, each gradient step would also move the target. The __target network__ $Q(s,a;\theta^{-})$, a copy of the Q-network, computes the targets instead. Every $C$ steps, counted over all episodes, the algorithm copies the Q-network into the target network, $\theta^{-}\gets\theta$.

Between copies of the target network, training is then close to a regression on random examples with fixed labels. The DQN algorithm puts the Q-network, the loss, and both stabilizers together.

### Algorithm
The DQN algorithm runs in episodes, like Q-learning.

__Initialize__: Given a Q-network $Q(s,a;\theta)$, set $\theta$ to zeros for a linear network or to small random values for a deep network, and copy the parameters to the target network, $\theta^{-}\gets\theta$. Start with an empty replay buffer $\mathcal{D}$ and a total step counter $\tau\gets 0$. 

Hyperparameters: the discount factor $0\leq\gamma<1$, the learning rate $\alpha>0$, the exploration probability $\epsilon$ of each episode, the number of episodes $N$, the maximum number of steps per episode $t_{\max}$, the buffer size $M$, the warm-up size $N_{\text{warm}}$ (the number of transitions the buffer must hold before training begins), the mini-batch size $N_{\text{batch}}$, and the target-network copy interval $C$.

For each of the $N$ episodes __do__:
1. Choose a starting state $s$ that is not absorbing, and set the episode step counter $t\gets 0$.
2. While $s$ is not absorbing and $t<t_{\max}$ __do__:
    1. Choose the action $a$: draw a random number $\rho$ uniformly from $[0,1]$.
        - If $\rho\leq\epsilon$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q(s,a^{\prime};\theta)$ (exploitation).
    2. Take action $a$, and observe the reward $r$, the next state $s^{\prime}$, and the done flag $d$ from the _world_, the simulation or system the agent acts on, where $d=1$ if $s^{\prime}$ is absorbing and $d=0$ otherwise.
    3. Store the transition $(s,a,r,s^{\prime},d)$ in the replay buffer $\mathcal{D}$, and remove the oldest transition if $\mathcal{D}$ holds more than $M$.
    4. Update the counters, $t\gets t+1$ and $\tau\gets\tau+1$.
    5. If $\mathcal{D}$ holds at least $N_{\text{warm}}$ transitions __then__ take one training step:
        1. Draw a mini-batch of $N_{\text{batch}}$ transitions $(s_{i},a_{i},r_{i},s_{i}^{\prime},d_{i})$, $i=1,\dots,N_{\text{batch}}$, uniformly at random with replacement from $\mathcal{D}$.
        2. Compute the target of each transition with the _target network_, $y_{i}\gets r_{i}+\gamma\,(1-d_{i})\max_{a^{\prime}\in\mathcal{A}}Q(s_{i}^{\prime},a^{\prime};\theta^{-})$.
        3. Compute the loss on the actions that were actually taken:
           $$\mathcal{L}(\theta)=\frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\left(y_{i}-Q(s_{i},a_{i};\theta)\right)^{2}.$$
        4. Take a _single_ gradient descent step on the loss, $\theta\gets\theta-\alpha\nabla_{\theta}\mathcal{L}(\theta)$.
            - _Why only a single step?_ Many steps on one small mini-batch can overfit its few transitions and undo what the network learned from the rest of the buffer.
    6. If $\tau$ is a multiple of $C$ __then__ copy the Q-network to the target network, $\theta^{-}\gets\theta$.
    7. Update the state, $s\gets s^{\prime}$.

__Extract Policy__: For any state $s$, compute $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q(s,a;\theta)$.

The policy gives an action for any state, including states the agent never visited.

__Why random starting weights for a deep network?__ If all the weights start equal, the units of a hidden layer compute the same feature, get the same update, and stay equal. A linear network has fixed features, and zeros work.

In the example, a linear network with 36 basis functions and a constant feature trains for 3,000 episodes in about one second. The same algorithm trains a deep network, which needs a package for neural networks and much more time.

> __Example:__
> 
> [▶ Learn a dosing policy without a grid using a linear DQN](CHEME-5800-L12c-Example-LinearDQN-AdaptiveDosing-Fall-2026.ipynb). In this example, an agent learns to dose the patient of the dosing lab without rounding the tumor burden and the toxicity to grid levels. The agent's linear Q-network has 148 weights, where the dosing lab's Q-table has 6,724 entries. We compare its doses with value iteration on a much finer grid.

### Overestimation
The DQN target uses the largest estimated Q-value of the next state. Every estimate has some error, and the largest one is often an estimate whose error is positive. The target then tends to be too high on average, even when each estimate is right on average.

This upward bias is called __overestimation__. It spreads back to the values of earlier states, and it can make a worse action look like the best one.

The maximum uses the same estimates to choose the next action and to score it. __Double DQN__ ([van Hasselt, Guez, and Silver, 2016](https://arxiv.org/abs/1509.06461)) chooses with the Q-network and scores with the target network. Its target is given by:

$$
\begin{equation*}
y_{i} = r_{i} + \gamma\,(1-d_{i})\,Q\left(s_{i}^{\prime},\;\arg\max_{a^{\prime}\in\mathcal{A}}Q(s_{i}^{\prime},a^{\prime};\theta);\;\theta^{-}\right).
\end{equation*}
$$

The target network is a recent copy of the Q-network, and the two share part of their errors. Double DQN can reduce overestimation, but it does not always remove it.

___

## Lab
In the [lab](../L12d/CHEME-5800-L12d-Lab-DoubleDQN-Fall-2026.ipynb), we'll measure overestimation and watch it make Q-learning and DQN prefer the worse action on a small problem with two states. We'll then test two fixes that choose with one estimate and score with another: Double Q-learning, which keeps two Q-tables that score each other's choices, and Double DQN.

___

## Summary
In this lecture, we replaced the Q-table with a Q-network and built deep Q-learning around it.

> __Key Takeaways:__
>
> * __A Q-network replaces the table:__ A table needs one entry per state and action and learns nothing about a state from its neighbors. A Q-network computes every action value from features of the state, with a number of parameters that does not grow with the number of states.
>
> * __Deep Q-learning trains the network by regression on bootstrap targets:__ Each target is the reward plus the discounted best value of the next state under the target network, or the reward alone when the next state is absorbing. For a linear network, one gradient step adds each prediction error times the features to the weights of the action taken.
>
> * __The replay buffer and the target network stabilize training:__ Random mini-batches from the replay buffer mix transitions from many episodes and reuse each one many times. The target network holds the targets fixed between copies, while the gradient steps update the Q-network.

A linear network with well-chosen features is often enough for a problem with a few state variables. A deep network learns its own features, which lets the same algorithm work from raw inputs such as images.

___